# Recognising knee rehabilitation exercises from sensor data

Design project at Chalmers by Linnea Person, Leo Barberan and Benjamin Houshmand.

We built a sensor setup for a medical knee brace: a rotary encoder at the joint
(knee angle and angular velocity) and two IMUs (accelerometer + gyroscope). The
goal is to tell from the signals which of five rehabilitation exercises is being done.

The notebook compares a 1D CNN, a bidirectional LSTM and a CNN + LSTM hybrid on
different sensor combinations, and at the end a reduced problem with three exercises.

The recordings are not included in the repository (see the README for the file format).

## Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_DIR = "data/train"
TEST_DIR = "data/test"
TRAIN_DIR_3EX = "data/train_3ex"
TEST_DIR_3EX = "data/test_3ex"

WINDOW_SIZE = 256
STEP_SIZE = 128
EPOCHS = 20

## Preprocessing

Every recording is a CSV named `Exercise<e>Setup<s>Set<n>.csv`. It starts with a short
calibration phase where the leg is held still; the mean encoder angle in that phase is
used as the zero angle. Only the rows of the exercise phase are kept, and every channel
is standardised per recording.

In [ ]:
PHASE_CALIBRATION = {"Calibrate", "Calibrated"}
PHASE_EXERCISE = {"Exercise"}


def parse_filename(path):
    fname = os.path.basename(path)
    if fname.startswith("._"):  # macOS metadata files inside the zips
        return None
    m = re.match(r"Exercise(\d+)Setup(\d+)Set(\d+)\.csv", fname)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))


def preprocess_file(df, feature_cols):
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()
    if "phase" not in df.columns:
        raise ValueError("No 'phase' column found")

    cal_df = df[df["phase"].isin(PHASE_CALIBRATION)]
    if cal_df.empty:
        raise ValueError("No calibration rows found in this file")
    angle_offset = pd.to_numeric(cal_df["EncoderAngle_deg"], errors="coerce").mean()
    df["EncoderAngleZeroed"] = pd.to_numeric(df["EncoderAngle_deg"], errors="coerce") - angle_offset

    ex_df = df[df["phase"].isin(PHASE_EXERCISE)].reset_index(drop=True)
    cols = feature_cols if "EncoderAngleZeroed" in feature_cols else ["EncoderAngleZeroed"] + feature_cols
    ex_df = ex_df[cols].apply(pd.to_numeric, errors="coerce")
    ex_df = ex_df.replace([np.inf, -np.inf], np.nan).dropna()

    for col in ex_df.columns:
        std = ex_df[col].std()
        ex_df[col] = ex_df[col] - ex_df[col].mean()
        if std > 0:
            ex_df[col] = ex_df[col] / std
    return ex_df

Sensor combinations we compare:

In [ ]:
ENCODER = ["EncoderAngleZeroed", "AngularVel_dps"]
IMU1_ACC = ["IMU1_AX_g", "IMU1_AY_g", "IMU1_AZ_g"]
IMU1_GYRO = ["IMU1_GX_dps", "IMU1_GY_dps", "IMU1_GZ_dps"]
IMU2_ACC = ["IMU2_AX_g", "IMU2_AY_g", "IMU2_AZ_g"]
IMU2_GYRO = ["IMU2_GX_dps", "IMU2_GY_dps", "IMU2_GZ_dps"]

FEATURE_SETS = {
    "full":      ENCODER + IMU1_ACC + IMU1_GYRO + IMU2_ACC + IMU2_GYRO,
    "encoder":   ENCODER,
    "imu1":      ENCODER + IMU1_ACC + IMU1_GYRO,
    "imu2":      ENCODER + IMU2_ACC + IMU2_GYRO,
    "imu1_gyro": ENCODER + IMU1_GYRO,
}

### Windows

Each recording is cut into windows of 256 samples with 50% overlap; the label of a
window is the exercise of the recording. Train and test recordings are kept in
separate folders. The validation set is a random 20% of the training windows.

In [ ]:
def create_windows(df, window_size, step_size, label, meta=None):
    data = df.to_numpy().astype(np.float32)
    windows = []
    for start in range(0, len(data) - window_size + 1, step_size):
        windows.append((data[start:start + window_size], label, dict(meta or {})))
    return windows


def collect_windows(root_dir, feature_cols, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    all_windows = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if not f.endswith(".csv"):
                continue
            info = parse_filename(f)
            if info is None:
                continue
            exercise_id, setup_id, set_id = info
            df_raw = pd.read_csv(os.path.join(root, f), low_memory=False)
            ex_df = preprocess_file(df_raw, feature_cols)
            meta = {"exercise_id": exercise_id, "setup_id": setup_id, "set_id": set_id, "fname": f}
            all_windows.extend(create_windows(ex_df, window_size, step_size, exercise_id, meta))
    return all_windows


def windows_to_arrays(windows):
    X = np.stack([w for w, _, _ in windows], axis=0)
    y = np.array([label for _, label, _ in windows], dtype=np.int64)
    meta = [m for _, _, m in windows]
    return X, y, meta

## Dataset and loaders

During training we add a bit of noise, random scaling and small time shifts.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.augment = augment

    def __len__(self):
        return self.X.shape[0]

    def _augment(self, x):
        if torch.rand(1) < 0.7:
            x = x + torch.randn_like(x) * 0.05
        if torch.rand(1) < 0.5:
            x = x * (0.9 + 0.2 * torch.rand(1))
        if torch.rand(1) < 0.3:
            shift = int(torch.randint(-3, 4, (1,)).item())
            x = torch.roll(x, shifts=shift, dims=0)
        return x

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            x = self._augment(x)
        return x.permute(1, 0), self.y[idx]  # [channels, time]


def make_loaders(feature_cols, train_dir=TRAIN_DIR, test_dir=TEST_DIR, first_label=1, batch_size=64):
    X_trainval, y_trainval, _ = windows_to_arrays(collect_windows(train_dir, feature_cols))
    X_test, y_test, _ = windows_to_arrays(collect_windows(test_dir, feature_cols))
    y_trainval = y_trainval - first_label  # labels start at 0
    y_test = y_test - first_label

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval)

    loaders = {
        "train": DataLoader(TimeSeriesDataset(X_train, y_train, augment=True), batch_size=batch_size, shuffle=True),
        "val":   DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=batch_size),
        "test":  DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=batch_size),
    }
    return loaders, X_train.shape[2], y_trainval

In [ ]:
data = {name: make_loaders(cols) for name, cols in FEATURE_SETS.items()}
num_classes = len(np.unique(data["full"][2]))

## Training and evaluation

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total


def eval_model(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            total_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total


def predict(model, loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds.append(model(X_batch.to(device)).argmax(1).cpu().numpy())
            true.append(y_batch.numpy())
    return np.concatenate(true), np.concatenate(preds)


def balanced_loss(y, n_classes):
    w = compute_class_weight(class_weight="balanced", classes=np.arange(n_classes), y=y)
    return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))


def fit_and_test(model, loaders, criterion, epochs=EPOCHS, lr=1e-3):
    """Train with Adam, print the progress and return the test confusion matrix."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, loaders["train"], optimizer, criterion)
        val_loss, val_acc = eval_model(model, loaders["val"], criterion)
        print(f"Epoch {epoch:02d} | "
              f"Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | "
              f"Val loss: {val_loss:.4f}, acc: {val_acc:.3f}")
    test_loss, test_acc = eval_model(model, loaders["test"], criterion)
    print(f"TEST | loss: {test_loss:.4f}, acc: {test_acc:.3f}")
    return confusion_matrix(*predict(model, loaders["test"]))

## 1D CNN

Three convolution layers (the last one dilated), then average and max pooling over
time and a small classifier. The loss uses class weights because the exercises do not
have the same number of windows.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, 64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 128, kernel_size=5, padding=4, dilation=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        feat = torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1).squeeze(-1)
        return self.fc(feat)


def run_cnn(name):
    loaders, n_features, y_trainval = data[name]
    model = CNN1D(n_features, num_classes).to(device)
    return fit_and_test(model, loaders, balanced_loss(y_trainval, num_classes))

### All sensors

In [ ]:
run_cnn("full")

Epoch 01 | Train loss: 0.7863, acc: 0.699 | Val loss: 0.3748, acc: 0.838
Epoch 02 | Train loss: 0.2943, acc: 0.885 | Val loss: 0.2199, acc: 0.908
Epoch 03 | Train loss: 0.1708, acc: 0.936 | Val loss: 0.1547, acc: 0.939
Epoch 04 | Train loss: 0.1484, acc: 0.943 | Val loss: 0.1604, acc: 0.932
Epoch 05 | Train loss: 0.1281, acc: 0.957 | Val loss: 0.0948, acc: 0.962
Epoch 06 | Train loss: 0.0963, acc: 0.966 | Val loss: 0.1050, acc: 0.966
Epoch 07 | Train loss: 0.0814, acc: 0.972 | Val loss: 0.0625, acc: 0.981
Epoch 08 | Train loss: 0.0515, acc: 0.983 | Val loss: 0.0500, acc: 0.978
Epoch 09 | Train loss: 0.0517, acc: 0.982 | Val loss: 0.0365, acc: 0.989
Epoch 10 | Train loss: 0.0496, acc: 0.983 | Val loss: 0.0732, acc: 0.965
Epoch 11 | Train loss: 0.0440, acc: 0.985 | Val loss: 0.0494, acc: 0.983
Epoch 12 | Train loss: 0.0390, acc: 0.986 | Val loss: 0.0527, acc: 0.984
Epoch 13 | Train loss: 0.0325, acc: 0.989 | Val loss: 0.0270, acc: 0.990
Epoch 14 | Train loss: 0.0315, acc: 0.989 | Val los

array([[ 73, 321,   5,  40,   0],
       [  0, 442,   0,  17,  25],
       [  5,  59, 263,  66,  19],
       [  0, 196,   0, 161,   0],
       [  0,  56,   0, 169,  61]])

### Encoder only

In [ ]:
run_cnn("encoder")

Epoch 01 | Train loss: 1.2618, acc: 0.417 | Val loss: 1.1261, acc: 0.474
Epoch 02 | Train loss: 1.1019, acc: 0.471 | Val loss: 1.0632, acc: 0.487
Epoch 03 | Train loss: 1.0782, acc: 0.477 | Val loss: 1.0270, acc: 0.530
Epoch 04 | Train loss: 1.0516, acc: 0.493 | Val loss: 1.0278, acc: 0.512
Epoch 05 | Train loss: 1.0321, acc: 0.498 | Val loss: 1.0095, acc: 0.525
Epoch 06 | Train loss: 1.0211, acc: 0.510 | Val loss: 1.0080, acc: 0.501
Epoch 07 | Train loss: 1.0091, acc: 0.514 | Val loss: 0.9873, acc: 0.515
Epoch 08 | Train loss: 0.9928, acc: 0.519 | Val loss: 0.9651, acc: 0.554
Epoch 09 | Train loss: 0.9695, acc: 0.539 | Val loss: 0.9785, acc: 0.540
Epoch 10 | Train loss: 0.9794, acc: 0.539 | Val loss: 0.9750, acc: 0.516
Epoch 11 | Train loss: 0.9510, acc: 0.547 | Val loss: 0.9638, acc: 0.547
Epoch 12 | Train loss: 0.9403, acc: 0.559 | Val loss: 0.9481, acc: 0.544
Epoch 13 | Train loss: 0.9404, acc: 0.555 | Val loss: 0.9305, acc: 0.559
Epoch 14 | Train loss: 0.9191, acc: 0.563 | Val los

array([[309,  18,   0,  74,  38],
       [294,  52,   0,  41,  97],
       [  0,   0, 398,  14,   0],
       [ 68,   0,   0, 229,  60],
       [ 62,   2,   0, 170,  52]])

### Encoder + IMU1

In [ ]:
run_cnn("imu1")

Epoch 01 | Train loss: 0.8617, acc: 0.610 | Val loss: 0.5079, acc: 0.736
Epoch 02 | Train loss: 0.4705, acc: 0.785 | Val loss: 0.4087, acc: 0.848
Epoch 03 | Train loss: 0.3508, acc: 0.855 | Val loss: 0.3087, acc: 0.876
Epoch 04 | Train loss: 0.2871, acc: 0.887 | Val loss: 0.2268, acc: 0.925
Epoch 05 | Train loss: 0.2358, acc: 0.904 | Val loss: 0.2201, acc: 0.917
Epoch 06 | Train loss: 0.2265, acc: 0.911 | Val loss: 0.2197, acc: 0.907
Epoch 07 | Train loss: 0.1824, acc: 0.925 | Val loss: 0.2470, acc: 0.919
Epoch 08 | Train loss: 0.1954, acc: 0.927 | Val loss: 0.1561, acc: 0.937
Epoch 09 | Train loss: 0.1594, acc: 0.933 | Val loss: 0.1529, acc: 0.939
Epoch 10 | Train loss: 0.1424, acc: 0.944 | Val loss: 0.1284, acc: 0.953
Epoch 11 | Train loss: 0.1285, acc: 0.954 | Val loss: 0.1264, acc: 0.956
Epoch 12 | Train loss: 0.1071, acc: 0.961 | Val loss: 0.1221, acc: 0.958
Epoch 13 | Train loss: 0.0997, acc: 0.965 | Val loss: 0.0864, acc: 0.971
Epoch 14 | Train loss: 0.0800, acc: 0.970 | Val los

array([[387,  52,   0,   0,   0],
       [ 13, 471,   0,   0,   0],
       [  2,   0, 346,  11,  53],
       [  0, 195,   0, 159,   3],
       [  0,   7,   0, 187,  92]])

### Encoder + IMU2

In [ ]:
run_cnn("imu2")

### Encoder + IMU1 gyroscope

In [ ]:
run_cnn("imu1_gyro")

Epoch 01 | Train loss: 1.1912, acc: 0.465 | Val loss: 0.9932, acc: 0.601
Epoch 02 | Train loss: 0.8949, acc: 0.607 | Val loss: 0.8176, acc: 0.616
Epoch 03 | Train loss: 0.7786, acc: 0.676 | Val loss: 0.7717, acc: 0.662
Epoch 04 | Train loss: 0.7313, acc: 0.689 | Val loss: 0.6994, acc: 0.709
Epoch 05 | Train loss: 0.6312, acc: 0.741 | Val loss: 0.6163, acc: 0.742
Epoch 06 | Train loss: 0.5950, acc: 0.759 | Val loss: 0.5297, acc: 0.791
Epoch 07 | Train loss: 0.5311, acc: 0.785 | Val loss: 0.5417, acc: 0.761
Epoch 08 | Train loss: 0.5574, acc: 0.780 | Val loss: 0.5414, acc: 0.772
Epoch 09 | Train loss: 0.5059, acc: 0.802 | Val loss: 0.4716, acc: 0.814
Epoch 10 | Train loss: 0.4697, acc: 0.810 | Val loss: 0.4649, acc: 0.818
Epoch 11 | Train loss: 0.4479, acc: 0.821 | Val loss: 0.4536, acc: 0.820
Epoch 12 | Train loss: 0.4101, acc: 0.838 | Val loss: 0.4841, acc: 0.818
Epoch 13 | Train loss: 0.4069, acc: 0.839 | Val loss: 0.4724, acc: 0.808
Epoch 14 | Train loss: 0.3958, acc: 0.845 | Val los

array([[341,  42,   1,  46,   9],
       [ 75, 376,   0,  11,  22],
       [  5,   0, 389,  14,   4],
       [ 58,   7,   0, 261,  31],
       [ 57,  18,   0, 142,  69]])

## Bidirectional LSTM

A linear projection of the channels, a 2-layer bidirectional LSTM, and mean + max
pooling over time.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, num_features, num_classes, proj_dim=64, hidden_size=64,
                 num_layers=2, bidirectional=True, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(num_features, proj_dim)
        self.lstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Sequential(
            nn.Linear(2 * out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.input_proj(x.permute(0, 2, 1))  # [batch, time, features]
        out, _ = self.lstm(x)
        pooled = torch.cat([out.mean(dim=1), out.max(dim=1).values], dim=1)
        return self.fc(pooled)


def run_lstm(name):
    loaders, n_features, y_trainval = data[name]
    model = LSTMClassifier(n_features, num_classes).to(device)
    return fit_and_test(model, loaders, balanced_loss(y_trainval, num_classes))

### All sensors

In [ ]:
run_lstm("full")

Epoch 01 | Train loss: 1.1567, acc: 0.495 | Val loss: 0.5810, acc: 0.736
Epoch 02 | Train loss: 0.3804, acc: 0.843 | Val loss: 0.2655, acc: 0.879
Epoch 03 | Train loss: 0.2074, acc: 0.918 | Val loss: 0.2058, acc: 0.912
Epoch 04 | Train loss: 0.1426, acc: 0.949 | Val loss: 0.1420, acc: 0.934
Epoch 05 | Train loss: 0.1422, acc: 0.948 | Val loss: 0.1675, acc: 0.940
Epoch 06 | Train loss: 0.1210, acc: 0.956 | Val loss: 0.1088, acc: 0.952
Epoch 07 | Train loss: 0.0727, acc: 0.974 | Val loss: 0.0892, acc: 0.968
Epoch 08 | Train loss: 0.0554, acc: 0.979 | Val loss: 0.1059, acc: 0.959
Epoch 09 | Train loss: 0.0584, acc: 0.979 | Val loss: 0.0668, acc: 0.978
Epoch 10 | Train loss: 0.0585, acc: 0.983 | Val loss: 0.1332, acc: 0.945
Epoch 11 | Train loss: 0.0850, acc: 0.972 | Val loss: 0.0763, acc: 0.973
Epoch 12 | Train loss: 0.0298, acc: 0.992 | Val loss: 0.0648, acc: 0.980
Epoch 13 | Train loss: 0.0155, acc: 0.995 | Val loss: 0.0477, acc: 0.985
Epoch 14 | Train loss: 0.0207, acc: 0.994 | Val los

array([[124, 308,   2,   5,   0],
       [ 16, 416,   2,  41,   9],
       [ 34, 111, 113, 116,  38],
       [  0,  62,   1, 251,  43],
       [  0,  12,   0, 211,  63]])

### Encoder only

In [ ]:
run_lstm("encoder")

Epoch 01 | Train loss: 1.4508, acc: 0.316 | Val loss: 1.1726, acc: 0.425
Epoch 02 | Train loss: 1.1420, acc: 0.404 | Val loss: 1.0993, acc: 0.445
Epoch 03 | Train loss: 1.0987, acc: 0.442 | Val loss: 1.0575, acc: 0.476
Epoch 04 | Train loss: 1.0702, acc: 0.457 | Val loss: 1.0376, acc: 0.486
Epoch 05 | Train loss: 1.0521, acc: 0.471 | Val loss: 1.0466, acc: 0.469
Epoch 06 | Train loss: 1.0530, acc: 0.465 | Val loss: 1.0379, acc: 0.484
Epoch 07 | Train loss: 1.0449, acc: 0.481 | Val loss: 1.0300, acc: 0.473
Epoch 08 | Train loss: 1.0315, acc: 0.476 | Val loss: 1.0392, acc: 0.476
Epoch 09 | Train loss: 1.0481, acc: 0.476 | Val loss: 1.0331, acc: 0.496
Epoch 10 | Train loss: 1.0061, acc: 0.504 | Val loss: 1.0034, acc: 0.518
Epoch 11 | Train loss: 0.9868, acc: 0.515 | Val loss: 0.9872, acc: 0.491
Epoch 12 | Train loss: 0.9846, acc: 0.517 | Val loss: 0.9898, acc: 0.499
Epoch 13 | Train loss: 0.9669, acc: 0.533 | Val loss: 0.9847, acc: 0.517
Epoch 14 | Train loss: 0.9703, acc: 0.536 | Val los

array([[211,  65,   0,  57, 106],
       [212,  63,   0,  43, 166],
       [  0,   0, 398,  14,   0],
       [103,   3,   0, 171,  80],
       [ 66,   5,   0, 136,  79]])

### Encoder + IMU1

In [ ]:
run_lstm("imu1")

Epoch 01 | Train loss: 1.1677, acc: 0.459 | Val loss: 0.7758, acc: 0.593
Epoch 02 | Train loss: 0.6074, acc: 0.703 | Val loss: 0.4954, acc: 0.794
Epoch 03 | Train loss: 0.4403, acc: 0.794 | Val loss: 0.4099, acc: 0.833
Epoch 04 | Train loss: 0.3582, acc: 0.836 | Val loss: 0.3420, acc: 0.865
Epoch 05 | Train loss: 0.2791, acc: 0.877 | Val loss: 0.2846, acc: 0.871
Epoch 06 | Train loss: 0.2580, acc: 0.889 | Val loss: 0.3054, acc: 0.868
Epoch 07 | Train loss: 0.2345, acc: 0.897 | Val loss: 0.2525, acc: 0.905
Epoch 08 | Train loss: 0.2146, acc: 0.912 | Val loss: 0.3004, acc: 0.873
Epoch 09 | Train loss: 0.1915, acc: 0.924 | Val loss: 0.2037, acc: 0.927
Epoch 10 | Train loss: 0.1494, acc: 0.944 | Val loss: 0.1904, acc: 0.932
Epoch 11 | Train loss: 0.1404, acc: 0.946 | Val loss: 0.1873, acc: 0.929
Epoch 12 | Train loss: 0.1256, acc: 0.953 | Val loss: 0.2509, acc: 0.913
Epoch 13 | Train loss: 0.1478, acc: 0.941 | Val loss: 0.2380, acc: 0.919
Epoch 14 | Train loss: 0.1245, acc: 0.950 | Val los

array([[404,  32,   1,   2,   0],
       [  1, 483,   0,   0,   0],
       [  2,   1, 353,  23,  33],
       [  1, 146,   0, 208,   2],
       [  0,   0,   0, 232,  54]])

### Encoder + IMU2

In [ ]:
run_lstm("imu2")

Epoch 01 | Train loss: 1.3117, acc: 0.435 | Val loss: 0.7979, acc: 0.709
Epoch 02 | Train loss: 0.6005, acc: 0.763 | Val loss: 0.4260, acc: 0.840
Epoch 03 | Train loss: 0.3533, acc: 0.863 | Val loss: 0.3316, acc: 0.870
Epoch 04 | Train loss: 0.2805, acc: 0.890 | Val loss: 0.2331, acc: 0.909
Epoch 05 | Train loss: 0.1795, acc: 0.932 | Val loss: 0.1953, acc: 0.931
Epoch 06 | Train loss: 0.1494, acc: 0.945 | Val loss: 0.2257, acc: 0.927
Epoch 07 | Train loss: 0.1399, acc: 0.947 | Val loss: 0.1903, acc: 0.941
Epoch 08 | Train loss: 0.1108, acc: 0.958 | Val loss: 0.2352, acc: 0.917
Epoch 09 | Train loss: 0.0914, acc: 0.968 | Val loss: 0.1432, acc: 0.955
Epoch 10 | Train loss: 0.0602, acc: 0.981 | Val loss: 0.1059, acc: 0.968
Epoch 11 | Train loss: 0.1007, acc: 0.967 | Val loss: 0.2316, acc: 0.927
Epoch 12 | Train loss: 0.0679, acc: 0.976 | Val loss: 0.1126, acc: 0.966
Epoch 13 | Train loss: 0.0404, acc: 0.987 | Val loss: 0.0832, acc: 0.969
Epoch 14 | Train loss: 0.0385, acc: 0.989 | Val los

array([[147, 127,   4, 158,   3],
       [ 54, 201,   2, 171,  56],
       [ 71, 104, 129, 108,   0],
       [  0,  52,   0, 305,   0],
       [  2,  75,   2, 207,   0]])

### Encoder + IMU1 gyroscope

In [ ]:
run_lstm("imu1_gyro")

Epoch 01 | Train loss: 1.4129, acc: 0.339 | Val loss: 1.0913, acc: 0.494
Epoch 02 | Train loss: 1.0006, acc: 0.542 | Val loss: 0.9487, acc: 0.582
Epoch 03 | Train loss: 0.8512, acc: 0.621 | Val loss: 0.8501, acc: 0.619
Epoch 04 | Train loss: 0.7769, acc: 0.660 | Val loss: 0.7697, acc: 0.691
Epoch 05 | Train loss: 0.7001, acc: 0.695 | Val loss: 0.7386, acc: 0.689
Epoch 06 | Train loss: 0.6526, acc: 0.727 | Val loss: 0.6748, acc: 0.711
Epoch 07 | Train loss: 0.5384, acc: 0.780 | Val loss: 0.5563, acc: 0.770
Epoch 08 | Train loss: 0.5385, acc: 0.783 | Val loss: 0.5898, acc: 0.730
Epoch 09 | Train loss: 0.4729, acc: 0.803 | Val loss: 0.5302, acc: 0.804
Epoch 10 | Train loss: 0.4242, acc: 0.835 | Val loss: 0.4818, acc: 0.809
Epoch 11 | Train loss: 0.3913, acc: 0.851 | Val loss: 0.4991, acc: 0.801
Epoch 12 | Train loss: 0.3684, acc: 0.862 | Val loss: 0.4569, acc: 0.813
Epoch 13 | Train loss: 0.3875, acc: 0.854 | Val loss: 0.4572, acc: 0.824
Epoch 14 | Train loss: 0.3322, acc: 0.875 | Val los

array([[286,  75,   3,  54,  21],
       [ 48, 398,   0,  12,  26],
       [  9,   8, 373,   5,  17],
       [ 49,  20,   0, 193,  95],
       [ 51,  41,   2,  92, 100]])

## Hybrid CNN + LSTM

Encoder and IMU1 got the best results above, so here each of them gets its own small
CNN, the two feature maps are stacked and an LSTM runs over the result.

In [ ]:
class HybridEncoderIMU1(nn.Module):
    def __init__(self, num_classes, encoder_idx, imu1_idx, enc_channels=32,
                 imu_channels=64, lstm_hidden=64, lstm_layers=1, dropout=0.3):
        super().__init__()
        self.encoder_idx = encoder_idx
        self.imu1_idx = imu1_idx
        self.enc_conv = self._conv_block(len(encoder_idx), enc_channels)
        self.imu_conv = self._conv_block(len(imu1_idx), imu_channels)
        self.lstm = nn.LSTM(
            input_size=enc_channels + imu_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.fc = nn.Sequential(
            nn.Linear(4 * lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    @staticmethod
    def _conv_block(c_in, c_out):
        return nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=5, padding=2),
            nn.BatchNorm1d(c_out),
            nn.ReLU(),
            nn.Conv1d(c_out, c_out, kernel_size=5, padding=2),
            nn.BatchNorm1d(c_out),
            nn.ReLU(),
        )

    def forward(self, x):
        enc = self.enc_conv(x[:, self.encoder_idx, :])
        imu = self.imu_conv(x[:, self.imu1_idx, :])
        fused = torch.cat([enc, imu], dim=1).permute(0, 2, 1)
        out, _ = self.lstm(fused)
        pooled = torch.cat([out.mean(dim=1), out.max(dim=1).values], dim=1)
        return self.fc(pooled)


# channel order of the "imu1" feature set
encoder_idx = [0, 1]
imu1_idx = [2, 3, 4, 5, 6, 7]

In [ ]:
loaders, _, _ = data["imu1"]
model_hybrid = HybridEncoderIMU1(num_classes, encoder_idx, imu1_idx).to(device)
fit_and_test(model_hybrid, loaders, nn.CrossEntropyLoss())

Epoch 01 | Train loss: 1.0119, acc: 0.529 | Val loss: 0.6172, acc: 0.714
Epoch 02 | Train loss: 0.5728, acc: 0.711 | Val loss: 0.4429, acc: 0.811
Epoch 03 | Train loss: 0.4582, acc: 0.785 | Val loss: 0.3559, acc: 0.847
Epoch 04 | Train loss: 0.4028, acc: 0.814 | Val loss: 0.3439, acc: 0.840
Epoch 05 | Train loss: 0.3492, acc: 0.848 | Val loss: 0.3329, acc: 0.840
Epoch 06 | Train loss: 0.3268, acc: 0.863 | Val loss: 0.2315, acc: 0.917
Epoch 07 | Train loss: 0.2993, acc: 0.871 | Val loss: 0.1947, acc: 0.924
Epoch 08 | Train loss: 0.2641, acc: 0.891 | Val loss: 0.2115, acc: 0.911
Epoch 09 | Train loss: 0.2465, acc: 0.899 | Val loss: 0.1757, acc: 0.938
Epoch 10 | Train loss: 0.2187, acc: 0.912 | Val loss: 0.1896, acc: 0.925
Epoch 11 | Train loss: 0.2195, acc: 0.911 | Val loss: 0.1784, acc: 0.934
Epoch 12 | Train loss: 0.2211, acc: 0.913 | Val loss: 0.1646, acc: 0.941
Epoch 13 | Train loss: 0.2025, acc: 0.918 | Val loss: 0.1523, acc: 0.942
Epoch 14 | Train loss: 0.1980, acc: 0.926 | Val los

[[381  57   0   1   0]
 [ 59 424   0   1   0]
 [  0   0 370  16  26]
 [  0  65   0 290   2]
 [  0   0   1 198  87]]


## Three exercises

Finally the hybrid model on a refined dataset that only contains exercises 2, 3 and 4.

In [ ]:
loaders_3ex, _, y_3ex = make_loaders(FEATURE_SETS["imu1"], TRAIN_DIR_3EX, TEST_DIR_3EX, first_label=2)
model_3ex = HybridEncoderIMU1(3, encoder_idx, imu1_idx).to(device)
fit_and_test(model_3ex, loaders_3ex, balanced_loss(y_3ex, 3))

Epoch 01 | Train loss: 0.4822, acc: 0.832 | Val loss: 0.1024, acc: 0.962
Epoch 02 | Train loss: 0.0998, acc: 0.965 | Val loss: 0.1051, acc: 0.957
Epoch 03 | Train loss: 0.0816, acc: 0.970 | Val loss: 0.0441, acc: 0.982
Epoch 04 | Train loss: 0.0476, acc: 0.981 | Val loss: 0.0429, acc: 0.982
Epoch 05 | Train loss: 0.0392, acc: 0.984 | Val loss: 0.0271, acc: 0.993
Epoch 06 | Train loss: 0.0282, acc: 0.992 | Val loss: 0.0126, acc: 0.995
Epoch 07 | Train loss: 0.0451, acc: 0.983 | Val loss: 0.0352, acc: 0.985
Epoch 08 | Train loss: 0.0293, acc: 0.990 | Val loss: 0.0123, acc: 0.993
Epoch 09 | Train loss: 0.0325, acc: 0.989 | Val loss: 0.0354, acc: 0.988
Epoch 10 | Train loss: 0.0355, acc: 0.990 | Val loss: 0.0246, acc: 0.993
Epoch 11 | Train loss: 0.0185, acc: 0.993 | Val loss: 0.0153, acc: 0.995
Epoch 12 | Train loss: 0.0174, acc: 0.994 | Val loss: 0.0075, acc: 0.998
Epoch 13 | Train loss: 0.0130, acc: 0.995 | Val loss: 0.0105, acc: 0.995
Epoch 14 | Train loss: 0.0295, acc: 0.990 | Val los

array([[482,   0,   2],
       [  0, 350,  62],
       [106,   3, 248]])